In [ ]:
from transformers import pipeline

# load the zero-shot classifier (first run downloads the model — may take a minute)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["opportunity", "risk", "trend"]

text = "Lufthansa faces pilot strikes and rising fuel costs threatening its profits."
result = classifier(text, candidate_labels=labels)

print(result)

{'sequence': 'Lufthansa faces pilot strikes and rising fuel costs threatening its profits.', 'labels': ['a business risk, threat or problem', 'an industry trend or market shift', 'a business opportunity or positive development'], 'scores': [0.8235887289047241, 0.15647804737091064, 0.019933223724365234]}


In [5]:
top_label = result["labels"][0]    # 'risk'  (first = highest score)
top_score = result["scores"][0]    # 0.93    (its confidence)

print("Category:", top_label)
print("Confidence:", round(top_score, 2))

Category: a business risk, threat or problem
Confidence: 0.82


In [6]:
import json
documents = json.load(open("lufthansa_data.json", encoding="utf-8"))

labels = ["opportunity", "risk", "trend"]

# test on the first 5 real docs
for d in documents[:5]:
    result = classifier(d["text"], candidate_labels=labels)
    cat   = result["labels"][0]
    score = result["scores"][0]
    print(f"[{cat}] ({score:.2f})  {d['text'][:90]}")
    print()

[trend] (0.53)  Financial reports - Lufthansa Group Investor Relations. Financial reports 2024 Annual Repo

[trend] (0.46)  Financial reports & publications - Lufthansa Group Investor Relations. The Lufthansa Group

[risk] (0.38)  Financial Data - Lufthansa Technik. Three-quarters of revenue now comes from business with

[trend] (0.41)  Lufthansa - Annual Reports - CompaniesMarketCap.com. Annual Reports Annual Reports Half-ye

[trend] (0.60)  Lufthansa Group Posts Record Revenue, Profit Surge. Mar 6, 2026 · COLOGNE — The Lufthansa 



In [8]:
# load the sentiment model (your News Analyst tool)
sentiment_pipe = pipeline("sentiment-analysis",
                          model="distilbert-base-uncased-finetuned-sst-2-english")

# test category + sentiment together on the first 5 docs
for d in documents[:5]:
    text = d["text"]
    category = classifier(text, candidate_labels=labels)["labels"][0]
    sent     = sentiment_pipe(text)[0]      # [{'label': 'POSITIVE', 'score': 0.99}]
    print(f"[{category}]  [{sent['label']} {sent['score']:.2f}]  {text[:80]}")

[trend]  [POSITIVE 0.97]  Financial reports - Lufthansa Group Investor Relations. Financial reports 2024 A
[trend]  [POSITIVE 0.99]  Financial reports & publications - Lufthansa Group Investor Relations. The Lufth
[risk]  [NEGATIVE 0.99]  Financial Data - Lufthansa Technik. Three-quarters of revenue now comes from bus
[trend]  [NEGATIVE 0.93]  Lufthansa - Annual Reports - CompaniesMarketCap.com. Annual Reports Annual Repor
[trend]  [POSITIVE 1.00]  Lufthansa Group Posts Record Revenue, Profit Surge. Mar 6, 2026 · COLOGNE — The 
